# Final dataset: Hyperion, EMIT and EnMAP pairs in one dataset (Member 1, notebook 4)

The three sensor notebooks (Hyperion `01b_Download_PiLOTl.ipynb`, `EMIT_LANDSAT_FINAL.ipynb`, `ENMAP_LANDSAT_FINAL.ipynb`) made the
**422 PASS pairs** (Hyperion 133, EMIT 258, EnMAP 31, QC v2.3) in the three `PASSED-PAIRS` folders. This notebook does **not** repeat
that work. It starts from those folders and finishes Member 1's remaining tasks:

| Member 1 task | done here by |
|---|---|
| DEM, land cover and other GIS data | **A1**: SRTM elevation / slope / aspect, GHSL built-up share, ESA WorldCover 2021, Dynamic World (scene year) |
| LST, NDVI, NDBI, NDWI | **A1**: indices from each pair's own Landsat (+ NDVI from the hyperspectral bands); LST is already in every Landsat file |
| final aligned dataset for AI | **A1** (same grid, pixel for pixel) + **B1**: one index of all pairs + train / val / test split by cell |
| EDA | **B2**: figures and tables across sensors, climate groups and splits |
| handover | **B3**: `README_DATASET.txt`, column dictionary, message for Member 2 |

## Output on Drive
```
MEMBER1___PREPROCESSING/
  HYPERION-LANDSAT/PASSED-PAIRS/<N>/   hyperion_reflectance.tif, landsat_<id>.tif, pair_info.json   (unchanged)
                                        aux_layers.tif   NEW (A1)
  EMIT-LANDSAT/PASSED-PAIRS/<N>/       emit_reflectance.tif, ...  + aux_layers.tif NEW
  ENMAP-LANDSAT/PASSED-PAIRS/<N>/      enmap_reflectance.tif, ... + aux_layers.tif NEW
  FINAL-DATASET/
    dataset_index.csv          one row per pair (all sensors): paths, dates, cell, climate, QC numbers, layer statistics, split (B1)
    dataset_index_columns.csv  what every column means (B3)
    cell_built_change.csv      GHSL built-up share 2015 and 2020 per cell (B1)
    README_DATASET.txt         for Member 2 (B3)
    eda/                       figures (PNG), tables (CSV), pixel sample (parquet) (A1, B2)
```

`aux_layers.tif` (float32, NaN = no data, **same grid as the hyperspectral file and the Landsat file**), 12 bands:

| # | band | meaning |
|---|---|---|
| 1 | NDVI | (NIR − red) / (NIR + red), Landsat, clear pixels only |
| 2 | NDBI | (SWIR1 − NIR) / (SWIR1 + NIR), Landsat, clear pixels only |
| 3 | NDWI | (green − NIR) / (green + NIR) (McFeeters), Landsat, clear pixels only |
| 4 | MNDWI | (green − SWIR1) / (green + SWIR1) (Xu), Landsat, clear pixels only |
| 5 | NDVI_hs | NDVI from the hyperspectral bands (mean of 636–673 nm and 851–879 nm), valid hyperspectral pixels |
| 6 | landsat_clear | 1 = clear Landsat pixel (same rule as the QC), 0 = not clear |
| 7 | elevation_m | SRTM 1 arc-second (bilinear) |
| 8 | slope_deg | slope from SRTM on the pair grid |
| 9 | aspect_deg | aspect from SRTM on the pair grid (0 = north, clockwise) |
| 10 | built_share | GHSL built-up surface share 0–1, epoch closest to the scene year (2020 at the latest) |
| 11 | landcover_worldcover | ESA WorldCover 2021 class code (most frequent 10 m class in the pixel) |
| 12 | landcover_dw | Dynamic World class (0 water … 8 snow), most frequent label within the scene date ± 6 months; NaN if none |

## Run order (Run all works)
| cell | what | time |
|---|---|---|
| **0 STATUS** | checks all pair folders and grids (read-only) | 1–2 min |
| **A1** | aux layers for every pair (Earth Engine + local); resumable | about 15–30 min the first time, seconds afterwards |
| **B1** | dataset index + GHSL change check + split | 2–3 min |
| **B2** | EDA figures and tables | 3–6 min the first time |
| **B3** | README, column dictionary, message for Member 2 | seconds |

**Adding pairs later** (e.g. more EnMAP scenes): run the sensor notebook (QC + handover), then run this notebook again.
A1 only makes the missing or outdated aux files; B1 keeps every existing cell in its split (the split is computed from the cell ID).

## Safety
| cell | writes | can it lose data? |
|---|---|---|
| 0 | nothing | no |
| A1 | `aux_layers.tif` in each PASS pair folder (new file), `FINAL-DATASET/eda/aux_examples.png` | no: the reflectance, Landsat and pair_info files are only read |
| B1, B2, B3 | files in `FINAL-DATASET/` only | no |

A1 remakes an aux file only when it is missing, made by an older `AUX_VERSION`, or older than the pair's reflectance / Landsat /
pair_info file (for example after a QC re-run). The EnMAP B3 and EMIT B2 handover cells were updated on 24 Oct to leave
`aux_layers.tif` in place (re-upload those notebooks to Colab before using them again).

## 0 STATUS
Read-only. Expected on 24 Oct: HYPERION 133, EMIT 258, ENMAP 31 pairs, all complete and listed in `handover_pass.csv`;
`aux_done` 0 before A1 has run. In the grid table, `same_grid` should be True for every example, `landsat_band8` is `QA_PIXEL` for
Hyperion and `clear_frac` for EMIT and EnMAP; pixel size about 30 m (Hyperion, EnMAP; UTM) and about 0.00054 deg (EMIT, lat/lon, about 60 m).

In [ ]:
# 0. STATUS: what the final-dataset step sees (read-only; changes nothing). Run any time.
import os, glob, json
import pandas as pd, rasterio

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; FINAL = f"{BASE}/FINAL-DATASET"
SENSORS = {"HYPERION": ("HYPERION-LANDSAT", "hyperion_reflectance.tif", 30),
           "EMIT": ("EMIT-LANDSAT", "emit_reflectance.tif", 60),
           "ENMAP": ("ENMAP-LANDSAT", "enmap_reflectance.tif", 30)}
pd.set_option("display.width", 220)
pair_dirs = lambda out: sorted([d for d in glob.glob(f"{out}/*") if os.path.isdir(d) and os.path.basename(d).isdigit()],
                               key=lambda d: int(os.path.basename(d)))

# ---- 1) every PASS pair folder ----
rows, grids = [], []
for s, (folder, refl, res) in SENSORS.items():
    out = f"{BASE}/{folder}/PASSED-PAIRS"; dirs = pair_dirs(out)
    csv = f"{out}/handover_pass.csv"
    listed = set(pd.read_csv(csv)["pair"].astype(int)) if os.path.exists(csv) else set()
    if not dirs: print(f"{s}: no pair folders found in {out}")
    for d in dirs:
        no = int(os.path.basename(d)); fs = set(os.listdir(d))
        nls = sum(f.startswith("landsat_") and f.endswith(".tif") for f in fs)
        rows.append({"sensor": s, "pair": no, "complete": refl in fs and "pair_info.json" in fs and nls == 1,
                     "in_csv": no in listed, "aux": "aux_layers.tif" in fs})
    for d in dirs[:2]:                                                   # grids of two example pairs per sensor (headers only)
        lf = glob.glob(f"{d}/landsat_*.tif")
        with rasterio.open(f"{d}/{refl}") as a, rasterio.open(lf[0]) as b:
            grids.append({"sensor": s, "pair": int(os.path.basename(d)), "crs": a.crs.to_string()[:24],
                          "pixel": tuple(round(abs(v), 6) for v in a.res), "size": f"{a.width} x {a.height}", "hs_bands": a.count,
                          "landsat_bands": b.count, "same_grid": a.shape == b.shape and a.transform.almost_equals(b.transform),
                          "landsat_band8": b.descriptions[7] if b.count > 7 else None})
st = pd.DataFrame(rows, columns=["sensor", "pair", "complete", "in_csv", "aux"])
summ = st.groupby("sensor").agg(pairs=("pair", "size"), complete=("complete", "sum"), in_handover_csv=("in_csv", "sum"),
                                aux_done=("aux", "sum"))
print("PASSED-PAIRS per sensor:\n" + summ.to_string())
print(f"TOTAL: {len(st)} pairs | complete {int(st['complete'].sum())} | listed in handover_pass.csv {int(st['in_csv'].sum())}"
      f" | with aux_layers.tif {int(st['aux'].sum())}")
bad = st[~st["complete"] | ~st["in_csv"]]
if len(bad): print("\nPROBLEMS (incomplete folder or not in handover_pass.csv):\n" + bad.to_string(index=False))
print("\nexample grids:\n" + pd.DataFrame(grids).to_string(index=False))

# ---- 2) FINAL-DATASET folder ----
ip = f"{FINAL}/dataset_index.csv"
n_idx = len(pd.read_csv(ip)) if os.path.exists(ip) else 0
n_png = len(glob.glob(f"{FINAL}/eda/*.png"))
print(f"\nFINAL-DATASET: index rows {n_idx} | EDA figures {n_png} | README {os.path.exists(f'{FINAL}/README_DATASET.txt')}")
n_aux = int(st["aux"].sum())
nxt = ("A1 (aux layers)" if n_aux < len(st) else "B1 (index + split)" if n_idx != len(st)
       else "B2 (EDA)" if n_png < 6 else "B3 (handover)" if not os.path.exists(f"{FINAL}/README_DATASET.txt") else "nothing: all done")
print("NEXT:", nxt)

# PART A: layers for every pair
## A1 Aux layers (`aux_layers.tif` in every PASS pair folder)
For each pair: indices from the pair's **own** Landsat bands (same date as the hyperspectral scene), NDVI from the hyperspectral bands,
the Landsat clear mask, and five Earth Engine layers requested **on exactly the pair's grid** (`computePixels`, as in EnMAP A4).
Pairs run 6 at a time. Statistics per pair (mean LST, NDVI, built-up share, land-cover shares, LST–NDVI correlation, ...) are stored in
the file's tags and used by B1. Ends with a summary per sensor and an example figure (the most built-up pair of each sensor).

**What to expect:** `made` = 422 the first time (0 on later runs, `current` = 422), `FAIL` = 0. Median LST–NDVI Spearman ρ negative
(vegetation is cooler), LST–NDBI positive. `dw_valid_frac` near 0 for Hyperion scenes before mid-2015 (Dynamic World starts June 2015).
Design choices are labelled in the settings block.

In [ ]:
# A1. AUX LAYERS for every PASS pair -> PASSED-PAIRS/<N>/aux_layers.tif, on exactly the grid of the hyperspectral file.
#     Indices from the pair's own Landsat (+ NDVI from the hyperspectral bands), Landsat clear mask, and Earth Engine layers:
#     SRTM elevation / slope / aspect, GHSL built-up share (epoch closest to the scene year), ESA WorldCover 2021,
#     Dynamic World (scene date +/- 6 months). Adds one file per pair; never changes the delivered files. Resumable.
import os, re, glob, json, time, warnings
import numpy as np, pandas as pd, rasterio, ee
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.stats import spearmanr
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
try: ee.Initialize(project="hyperspectral-thermal-511303")
except Exception: ee.Authenticate(); ee.Initialize(project="hyperspectral-thermal-511303")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; FINAL = f"{BASE}/FINAL-DATASET"; EDA = f"{FINAL}/eda"
os.makedirs(EDA, exist_ok=True)
SENSORS = {"HYPERION": ("HYPERION-LANDSAT", "hyperion_reflectance.tif", 30),
           "EMIT": ("EMIT-LANDSAT", "emit_reflectance.tif", 60),
           "ENMAP": ("ENMAP-LANDSAT", "enmap_reflectance.tif", 30)}
pair_dirs = lambda out: sorted([d for d in glob.glob(f"{out}/*") if os.path.isdir(d) and os.path.basename(d).isdigit()],
                               key=lambda d: int(os.path.basename(d)))
pd.set_option("display.width", 220)

# ---- settings (design choices) ----
AUX_VERSION = 1             # bump to rebuild every aux file after a rule change
FORCE = False               # True = rebuild all aux files now
WORKERS = 6                 # pairs processed in parallel (the Earth Engine requests run concurrently)
CLEAR_FRAC_MIN = 0.8        # EMIT / EnMAP Landsat chips: clear if >= 80 % of the 30 m Landsat pixels were clear (same as QC v2.3)
RED_WIN, NIR_WIN = (636, 673), (851, 879)   # hyperspectral bands averaged over the Landsat OLI red / NIR ranges (same as the QC)
DW_HALF_DAYS = 183          # Dynamic World: most frequent class within the scene date +/- 6 months
GHSL_MAX_EPOCH = 2020       # GHSL epochs after 2020 are projections, not observations -> not used
HALF = 0.035                # cell half-width (deg), same 0.07 deg cell as in the screening
AUX_BANDS = ["NDVI", "NDBI", "NDWI", "MNDWI", "NDVI_hs", "landsat_clear", "elevation_m", "slope_deg", "aspect_deg",
             "built_share", "landcover_worldcover", "landcover_dw"]
EE_NAMES = AUX_BANDS[6:]
WC_NAMES = {10: "tree", 20: "shrub", 30: "grass", 40: "crop", 50: "built", 60: "bare", 70: "snow", 80: "water",
            90: "wetland", 95: "mangrove", 100: "moss"}

# ---- Earth Engine sources ----
DEM = ee.Image("USGS/SRTMGL1_003").select("elevation")
GHSL_IC = "JRC/GHSL/P2023A/GHS_BUILT_S"
try:
    ids = ee.ImageCollection(GHSL_IC).aggregate_array("system:index").getInfo()
    EPOCHS = sorted(int(i) for i in ids if str(i).isdigit() and int(i) <= GHSL_MAX_EPOCH)
except Exception: EPOCHS = list(range(1975, GHSL_MAX_EPOCH + 1, 5))
WC = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
DW_EMPTY = ee.Image.constant(0).toByte().rename("label").updateMask(ee.Image.constant(0))   # keeps the collection non-empty
print(f"GHSL epochs used: {EPOCHS}")

def norm_date(x):                    # '2014-03-05 05:12:00', '20230415' or '2023-04-15' -> '2023-04-15'
    m = re.search(r"(\d{4})-?(\d{2})-?(\d{2})", str(x))
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}" if m else None
def nd(a, b):
    with np.errstate(divide="ignore", invalid="ignore"): x = (a - b) / (a + b)
    return np.where(np.isfinite(x), np.clip(x, -1, 1), np.nan).astype("float32")
def wavelengths(src):                # band descriptions start with the wavelength, e.g. '650 nm (B032)' or '652.3 nm (EnMAP b045)'
    return np.array([float(m.group(1)) if (m := re.match(r"\s*([\d.]+)\s*nm", d or "")) else np.nan for d in src.descriptions])
def band_mean(src, wl, win):         # mean reflectance of the bands inside a wavelength window (nearest band if none)
    ix = np.where((wl >= win[0]) & (wl <= win[1]))[0]
    if len(ix) == 0: ix = [int(np.nanargmin(np.abs(wl - np.mean(win))))]
    A = src.read([int(i) + 1 for i in ix]).astype("float32"); A[A == -32768] = np.nan
    return np.where(np.isfinite(A).all(axis=0), A.mean(axis=0) / 10000.0, np.nan).astype("float32")
def landsat_layers(path, sensor):    # band 8 is QA_PIXEL (Hyperion pairs) or clear_frac (EMIT / EnMAP pairs)
    with rasterio.open(path) as s: L = s.read().astype("float32"); last = (s.descriptions[7] or "") if s.count > 7 else ""
    if L.shape[0] < 8: raise ValueError(f"Landsat file has {L.shape[0]} bands, expected 8")
    if "QA" in last.upper() or (sensor == "HYPERION" and "clear" not in last.lower()):
        qa = np.nan_to_num(L[7]).astype("int64"); clear = ((qa & 0b11110) == 0) & (qa != 0)     # QA bits 1-4 = 0
    else: clear = np.nan_to_num(L[7]) >= CLEAR_FRAC_MIN
    clear &= np.isfinite(L[2]) & np.isfinite(L[3])
    return {"green": L[1], "red": L[2], "nir": L[3], "swir1": L[4], "lst": L[6], "clear": clear}
def ee_layers(crs, tf, w, h, date, lon, lat):
    epsg = crs.to_epsg(); crs_str = f"EPSG:{epsg}" if epsg else crs.to_wkt()
    grid = {"dimensions": {"width": w, "height": h}, **({"crsCode": crs_str} if epsg else {"crsWkt": crs_str}),
            "affineTransform": {"scaleX": tf.a, "shearX": tf.b, "translateX": tf.c, "shearY": tf.d, "scaleY": tf.e, "translateY": tf.f}}
    year = int(date[:4]); ep = min(EPOCHS, key=lambda e: (abs(e - year), -e))
    terr = ee.Terrain.products(DEM.resample("bilinear"))                       # slope / aspect computed on the pair grid
    built = ee.Image(f"{GHSL_IC}/{ep}").select("built_surface").divide(10000).resample("bilinear")   # m2 per 100 m cell -> share
    proj = ee.Projection(crs_str, [tf.a, tf.b, tf.c, tf.d, tf.e, tf.f])
    wc = WC.reduceResolution(reducer=ee.Reducer.mode(), maxPixels=1024).reproject(proj)               # most frequent 10 m class
    t = pd.Timestamp(date); rect = ee.Geometry.Rectangle([lon - HALF, lat - HALF, lon + HALF, lat + HALF], None, False)
    dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(rect)
          .filterDate((t - pd.Timedelta(days=DW_HALF_DAYS)).strftime("%Y-%m-%d"), (t + pd.Timedelta(days=DW_HALF_DAYS)).strftime("%Y-%m-%d"))
          .select("label").merge(ee.ImageCollection([DW_EMPTY])).mode())
    img = (ee.Image.cat([terr.select("elevation"), terr.select("slope"), terr.select("aspect"), built, wc, dw])
             .rename(EE_NAMES).toFloat().unmask(-9999))
    for attempt in range(3):
        try: arr = ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY", "grid": grid}); break
        except Exception:
            if attempt == 2: raise
            time.sleep(5 * (attempt + 1))
    A = np.stack([arr[n] for n in EE_NAMES]).astype("float32"); A[A <= -9998] = np.nan
    return A, ep
f3 = lambda x: None if x is None or not np.isfinite(x) else round(float(x), 3)
def pair_stats(L, ndvi, ndbi, ndwi, ndvi_hs, G, joint):
    lst = L["lst"]; j = joint & np.isfinite(lst) & (lst > 200) & (lst < 360); n = int(j.sum())
    elev, slope, _, built, wc, dw = G
    st = {"n_px": int(j.size), "n_joint_px": n, "joint_frac": f3(n / j.size), "landsat_clear_frac": f3(L["clear"].mean()),
          "elev_mean": f3(np.nanmean(elev)), "slope_mean": f3(np.nanmean(slope)), "built_share_mean": f3(np.nanmean(built)),
          "dw_valid_frac": f3(np.isfinite(dw).mean())}
    wcv = wc[np.isfinite(wc)]
    for k, nme in WC_NAMES.items(): st[f"wc_{nme}"] = f3((wcv == k).mean()) if wcv.size else None
    if n >= 100:
        lc = lst[j] - 273.15
        st.update({"lst_c_mean": f3(lc.mean()), "lst_c_p10": f3(np.percentile(lc, 10)), "lst_c_p90": f3(np.percentile(lc, 90)),
                   "ndvi_mean": f3(np.nanmean(ndvi[j])), "ndbi_mean": f3(np.nanmean(ndbi[j])), "ndwi_mean": f3(np.nanmean(ndwi[j])),
                   "ndvi_hs_mean": f3(np.nanmean(ndvi_hs[j]))})
        ii = np.flatnonzero(j); ii = np.random.default_rng(0).choice(ii, min(len(ii), 20000), replace=False)
        for nme, x in (("ndvi", ndvi), ("ndbi", ndbi)):
            v, y = x.ravel()[ii], lst.ravel()[ii]; ok = np.isfinite(v)
            st[f"rho_lst_{nme}"] = f3(spearmanr(y[ok], v[ok])[0]) if ok.sum() >= 100 else None
    return st
def is_current(aux, sources):
    if FORCE or not os.path.exists(aux): return False
    try:
        with rasterio.open(aux) as s: t = s.tags()
        if int(t.get("aux_version", 0)) != AUX_VERSION: return False
    except Exception: return False
    return os.path.getmtime(aux) >= max(os.path.getmtime(p) for p in sources)

def make_aux(job):
    s, no, d, refl = job
    rf, ip, aux = f"{d}/{refl}", f"{d}/pair_info.json", f"{d}/aux_layers.tif"; lsf = glob.glob(f"{d}/landsat_*.tif")
    if not (os.path.exists(rf) and os.path.exists(ip) and len(lsf) == 1):
        return {"sensor": s, "pair": no, "result": "FAIL", "why": f"files missing ({len(lsf)} landsat files)"}
    if is_current(aux, [rf, lsf[0], ip]): return {"sensor": s, "pair": no, "result": "current"}
    t0 = time.time()
    try:
        info = json.load(open(ip)); date = norm_date(info.get(f"{s.lower()}_date"))
        if not date: raise ValueError(f"no scene date in pair_info ({s.lower()}_date)")
        with rasterio.open(rf) as src:
            crs, tf, w, h = src.crs, src.transform, src.width, src.height
            wl = wavelengths(src)
            if not np.isfinite(wl).any(): raise ValueError("no wavelengths in the band descriptions")
            red_h, nir_h = band_mean(src, wl, RED_WIN), band_mean(src, wl, NIR_WIN)
        L = landsat_layers(lsf[0], s)
        if L["nir"].shape != (h, w): raise ValueError(f"Landsat grid {L['nir'].shape} differs from the hyperspectral grid {(h, w)}")
        c = L["clear"]; m = lambda x: np.where(c, x, np.nan).astype("float32")
        ndvi, ndbi, ndwi, mndwi = m(nd(L["nir"], L["red"])), m(nd(L["swir1"], L["nir"])), m(nd(L["green"], L["nir"])), m(nd(L["green"], L["swir1"]))
        ndvi_hs = nd(nir_h, red_h)
        G, ep = ee_layers(crs, tf, w, h, date, float(info["cell_lon"]), float(info["cell_lat"]))
        st = pair_stats(L, ndvi, ndbi, ndwi, ndvi_hs, G, c & np.isfinite(ndvi_hs))
        tmp = aux + ".tmp"
        with rasterio.open(tmp, "w", driver="GTiff", height=h, width=w, count=len(AUX_BANDS), dtype="float32", crs=crs, transform=tf,
                           nodata=np.nan, compress="deflate") as dst:
            dst.write(np.stack([ndvi, ndbi, ndwi, mndwi, ndvi_hs, c.astype("float32"), *G]).astype("float32"))
            for k, nme in enumerate(AUX_BANDS, start=1): dst.set_band_description(k, nme)
            dst.update_tags(aux_version=AUX_VERSION, sensor=s, pair_no=no, scene_date=date, ghsl_epoch=ep,
                            worldcover="ESA WorldCover v200 (2021), mode of 10 m classes", dynamic_world=f"label mode, {date} +/- {DW_HALF_DAYS} d",
                            landsat_clear_rule="QA_PIXEL bits 1-4 = 0" if s == "HYPERION" else f"clear_frac >= {CLEAR_FRAC_MIN}",
                            stats_json=json.dumps(st), created=time.strftime("%Y-%m-%d %H:%M"))
        os.replace(tmp, aux)
        return {"sensor": s, "pair": no, "result": "made", "sec": round(time.time() - t0, 1)}
    except Exception as e:
        return {"sensor": s, "pair": no, "result": "FAIL", "why": f"{type(e).__name__}: {str(e)[:200]}"}

# ---- run (parallel, resumable) ----
jobs = [(s, int(os.path.basename(d)), d, refl) for s, (folder, refl, _) in SENSORS.items() for d in pair_dirs(f"{BASE}/{folder}/PASSED-PAIRS")]
print(f"pairs: {len(jobs)} | aux version {AUX_VERSION} | force {FORCE} | workers {WORKERS}")
res, t_all = [], time.time()
with ThreadPoolExecutor(WORKERS) as ex:
    for k, f in enumerate(as_completed([ex.submit(make_aux, j) for j in jobs]), 1):
        r = f.result(); res.append(r)
        if r["result"] == "FAIL": print(f"  FAIL {r['sensor']} {r['pair']}: {r['why']}")
        n_made = sum(x["result"] == "made" for x in res)
        if r["result"] == "made" and n_made % 25 == 0: print(f"  made {n_made} | checked {k}/{len(jobs)} | {(time.time() - t_all) / 60:.1f} min")
R = pd.DataFrame(res)
print(f"\nthis run ({(time.time() - t_all) / 60:.1f} min):\n" + pd.crosstab(R["sensor"], R["result"]).to_string())

# ---- summary from all aux files (tags) ----
where = {(s, no): (d, refl) for s, no, d, refl in jobs}
S = []
for s, no, d, refl in jobs:
    if os.path.exists(f"{d}/aux_layers.tif"):
        with rasterio.open(f"{d}/aux_layers.tif") as x: t = x.tags()
        S.append({"sensor": s, "pair": no, "ghsl_epoch": int(t.get("ghsl_epoch", 0)), **json.loads(t.get("stats_json", "{}"))})
S = pd.DataFrame(S)
print(f"\npairs with an aux file: {len(S)} of {len(jobs)}")
if len(S):
    cols = [c for c in ["joint_frac", "lst_c_mean", "ndvi_mean", "ndvi_hs_mean", "ndbi_mean", "built_share_mean", "wc_built", "wc_tree",
                        "elev_mean", "slope_mean", "dw_valid_frac", "rho_lst_ndvi", "rho_lst_ndbi"] if c in S.columns]
    print("\nmedians per sensor:\n" + S.groupby("sensor")[cols].median().round(3).T.to_string())
    print("\nGHSL epoch used:", {k: v for k, v in S.groupby("sensor")["ghsl_epoch"].value_counts().items()})
    ok = S[["ndvi_mean", "ndvi_hs_mean"]].dropna() if "ndvi_hs_mean" in S.columns else pd.DataFrame()
    if len(ok) > 5: print(f"check: mean NDVI hyperspectral vs Landsat across pairs: r = {np.corrcoef(ok['ndvi_mean'], ok['ndvi_hs_mean'])[0, 1]:.3f} "
                          f"(expected > 0.9), median difference {np.median(ok['ndvi_hs_mean'] - ok['ndvi_mean']):+.3f}")
    if "rho_lst_ndvi" in S.columns:
        print(f"LST-NDVI Spearman < 0 in {100 * (S['rho_lst_ndvi'] < 0).mean():.0f} % of pairs | LST-NDBI > 0 in "
              f"{100 * (S['rho_lst_ndbi'] > 0).mean():.0f} % of pairs")

# ---- example figure: the most built-up pair of each sensor ----
WC_COL = {10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff", 50: "#fa0000", 60: "#b4b4b4", 70: "#f0f0f0", 80: "#0064c8",
          90: "#0096a0", 95: "#00cf75", 100: "#fae6a0"}            # official ESA WorldCover legend colours
codes = list(WC_COL)
cmap_wc = ListedColormap(list(WC_COL.values()))
norm_wc = BoundaryNorm([codes[0] - 5] + [(a + b) / 2 for a, b in zip(codes[:-1], codes[1:])] + [codes[-1] + 5], len(codes))
ex = S.sort_values("wc_built", ascending=False).groupby("sensor").head(1) if len(S) and "wc_built" in S.columns else pd.DataFrame()
if len(ex):
    fig, axs = plt.subplots(len(ex), 6, figsize=(20, 3.4 * len(ex)), squeeze=False, facecolor="#fcfcfb")
    for i, r in enumerate(ex.itertuples()):
        d, refl = where[(r.sensor, r.pair)]
        with rasterio.open(f"{d}/aux_layers.tif") as a: A = {n: a.read(k + 1) for k, n in enumerate(a.descriptions)}
        with rasterio.open(glob.glob(f"{d}/landsat_*.tif")[0]) as l: lst = l.read(7)
        panels = [("LST (deg C), Landsat", np.where(A["landsat_clear"] == 1, lst - 273.15, np.nan), "Reds", None, None),
                  ("NDVI", A["NDVI"], "BrBG", -0.8, 0.8), ("NDBI", A["NDBI"], "BrBG_r", -0.8, 0.8),
                  ("built-up share (GHSL)", A["built_share"], "Purples", 0, 1), ("WorldCover 2021", A["landcover_worldcover"], None, None, None),
                  ("elevation (m)", A["elevation_m"], "Greys", None, None)]
        for j, (title, img, cm, lo, hi) in enumerate(panels):
            ax = axs[i, j]; ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"{r.sensor} {r.pair}: {title}" if j == 0 else title, loc="left", fontsize=9)
            if not np.isfinite(img).any(): ax.text(0.5, 0.5, "no data", ha="center", transform=ax.transAxes); continue
            if cm is None: ax.imshow(img, cmap=cmap_wc, norm=norm_wc, interpolation="nearest"); continue
            im = ax.imshow(img, cmap=cm, interpolation="nearest", vmin=lo if lo is not None else np.nanpercentile(img, 2),
                           vmax=hi if hi is not None else np.nanpercentile(img, 98))
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    fig.text(0.01, -0.01, "WorldCover: " + ", ".join(f"{k} {v}" for k, v in WC_NAMES.items()) + "  (red = built, dark green = tree, blue = water)",
             fontsize=8, color="#52514e")
    plt.tight_layout(); fig.savefig(f"{EDA}/aux_examples.png", dpi=130, bbox_inches="tight"); plt.show()
print(f"\nNEXT: B1 (index + split) once FAIL = 0 and every pair has an aux file. Figure saved: {EDA}/aux_examples.png")

# PART B: one dataset
## B1 Dataset index + GHSL change check + train / val / test split
One row per PASS pair of all three sensors, built from `pair_info.json` and the aux-file statistics, with paths **relative to
`MEMBER1___PREPROCESSING/`** so the folder can be shared or moved. Adds the GHSL built-up share 2015 and 2020 per cell
(flag `built_growth_large` for scenes up to 2017 whose cell gained more than 10 percentage points: those cells changed between the
scene and the 2020/2021 data used in the screening). Then the split.

**Split (design choice):** by **cell**, never by pair (several dates of the same cell would otherwise leak between train and test).
The group of a cell comes from a hash of its ID, so it never changes when pairs are added later; a cell seen by two sensors is in one
group for both. Target 70 / 15 / 15 % of cells. `SPLIT_BY = "block_1deg"` keeps whole 1 x 1 degree blocks together (stricter, against
neighbouring cells; the cell prints how many neighbouring cells are split apart).

**What to expect:** 422 rows; no cell in two splits; roughly 70 / 15 / 15 % of pairs per split (exact shares vary because cells
have different numbers of pairs); `built_growth_large` for only a few Hyperion pairs.

In [ ]:
# B1. DATASET INDEX + GHSL CHANGE CHECK + TRAIN/VAL/TEST SPLIT -> FINAL-DATASET/dataset_index.csv (+ cell_built_change.csv)
import os, re, glob, json, hashlib, warnings
import numpy as np, pandas as pd, rasterio, ee
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
try: ee.Initialize(project="hyperspectral-thermal-511303")
except Exception: ee.Authenticate(); ee.Initialize(project="hyperspectral-thermal-511303")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; FINAL = f"{BASE}/FINAL-DATASET"; os.makedirs(FINAL, exist_ok=True)
SENSORS = {"HYPERION": ("HYPERION-LANDSAT", "hyperion_reflectance.tif", 30),
           "EMIT": ("EMIT-LANDSAT", "emit_reflectance.tif", 60),
           "ENMAP": ("ENMAP-LANDSAT", "enmap_reflectance.tif", 30)}
pair_dirs = lambda out: sorted([d for d in glob.glob(f"{out}/*") if os.path.isdir(d) and os.path.basename(d).isdigit()],
                               key=lambda d: int(os.path.basename(d)))
pd.set_option("display.width", 220)

# ---- settings (design choices) ----
SPLIT_BY = "cell_id"               # "cell_id" (agreed) or "block_1deg" (stricter: whole 1 x 1 deg blocks stay together)
SPLIT_FRAC = (0.70, 0.15, 0.15)    # train / val / test share of groups
GROWTH_PP, OLD_SCENE_YEAR = 10.0, 2017   # flag cells that gained > 10 percentage points built-up 2015 -> 2020 (scenes up to 2017)
HALF, GHSL_IC = 0.035, "JRC/GHSL/P2023A/GHS_BUILT_S"
QC_VERSION = "v2.3"

def norm_date(x):
    m = re.search(r"(\d{4})-?(\d{2})-?(\d{2})", str(x))
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}" if m else None
def norm_cell(x):                  # 12345, 12345.0 and '12345' -> '12345'
    s = str(x).strip(); return s[:-2] if re.fullmatch(r"-?\d+\.0", s) else s

# ---- 1) one row per pair ----
rows, problems = [], []
for s, (folder, refl, res) in SENSORS.items():
    for d in pair_dirs(f"{BASE}/{folder}/PASSED-PAIRS"):
        no = int(os.path.basename(d)); ip = f"{d}/pair_info.json"
        if not os.path.exists(ip): problems.append(f"{s} {no}: no pair_info.json"); continue
        info = json.load(open(ip)); q = info.get("final_qc") or {}; g = info.get("geometry") or {}
        if info.get("status") != "PASS" or info.get("qc_version") != QC_VERSION:
            problems.append(f"{s} {no}: status {info.get('status')}, qc {info.get('qc_version')} (expected PASS, {QC_VERSION})")
        lsf = sorted(glob.glob(f"{d}/landsat_*.tif")); auxp = f"{d}/aux_layers.tif"; t = {}
        if os.path.exists(auxp):
            with rasterio.open(auxp) as x: t = x.tags()
        else: problems.append(f"{s} {no}: no aux_layers.tif (run A1)")
        with rasterio.open(f"{d}/{refl}") as x: nb, w, h, crs = x.count, x.width, x.height, x.crs.to_string()
        date = norm_date(info.get(f"{s.lower()}_date")); fl = q.get("flags_outside_range") or ""
        rows.append({"dataset_id": f"{s}_{no:04d}", "sensor": s, "pair_no": no, "rel_dir": os.path.relpath(d, BASE),
                     "reflectance_file": refl, "landsat_file": os.path.basename(lsf[0]) if lsf else None,
                     "aux_file": "aux_layers.tif" if t else None, "res_m": res, "crs": crs, "width": w, "height": h, "n_bands": nb,
                     "scene_id": info.get(f"{s.lower()}_scene"), "scene_date": date,
                     "year": int(date[:4]) if date else None, "month": int(date[5:7]) if date else None,
                     "landsat_scene": info.get("landsat_scene"), "landsat_dt_days": info.get(f"landsat_minus_{s.lower()}_days"),
                     "cell_id": norm_cell(info.get("cell_id")), "cell_lon": info.get("cell_lon"), "cell_lat": info.get("cell_lat"),
                     "profile": info.get("profile"), "koppen_code": info.get("koppen_code"), "climate_group": info.get("climate_group"),
                     "qc_version": info.get("qc_version"), "coverage_joint": q.get("coverage_joint"),
                     "landsat_clear_frac_qc": q.get("landsat_clear_frac"), "ndvi_r_after": q.get("ndvi_r_after"),
                     "ndvi_gain": q.get("ndvi_gain"), "aod550": q.get("aod550"), "sun_elev": q.get("sun_elev"),
                     "across_offnadir": g.get("across_offnadir"),
                     "flag_blue": "blue" in fl, "flag_red": "red" in fl, "flag_swir1": "swir1" in fl,
                     "haze_mild": bool(q.get("haze_mild") or False), "haze_unchecked": "haze_unchecked" in fl,
                     "large_shift": bool(q.get("large_shift") or False),
                     "ghsl_epoch": int(t["ghsl_epoch"]) if t.get("ghsl_epoch") else None, **json.loads(t.get("stats_json", "{}"))})
idx = pd.DataFrame(rows)
print(f"(1) pairs: {len(idx)} | {idx['sensor'].value_counts().to_dict()} | with aux file: {int(idx['aux_file'].notna().sum())}")

# ---- 2) GHSL built-up share 2015 and 2020 per cell (cached in cell_built_change.csv) ----
cp = f"{FINAL}/cell_built_change.csv"
cc = pd.read_csv(cp, dtype={"cell_id": str}) if os.path.exists(cp) else pd.DataFrame(columns=["cell_id", "built_2015", "built_2020"])
cells = idx.drop_duplicates("cell_id")[["cell_id", "cell_lon", "cell_lat"]]
todo = cells[~cells["cell_id"].isin(cc["cell_id"])]
if len(todo):
    img = (ee.Image.cat([ee.Image(f"{GHSL_IC}/2015").select("built_surface"), ee.Image(f"{GHSL_IC}/2020").select("built_surface")])
             .divide(10000).rename(["built_2015", "built_2020"]))
    new = []
    for k in range(0, len(todo), 100):
        fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Rectangle([lo - HALF, la - HALF, lo + HALF, la + HALF], None, False), {"cell_id": c})
                                   for c, lo, la in todo.iloc[k:k + 100].itertuples(index=False)])
        got = img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=100).getInfo()["features"]
        new += [{"cell_id": f["properties"]["cell_id"], "built_2015": f["properties"].get("built_2015"),
                 "built_2020": f["properties"].get("built_2020")} for f in got]
    cc = pd.concat([cc, pd.DataFrame(new)], ignore_index=True) if len(cc) else pd.DataFrame(new); cc.to_csv(cp, index=False)
idx = idx.merge(cc[["cell_id", "built_2015", "built_2020"]], on="cell_id", how="left")
idx["built_growth_pp"] = (100 * (idx["built_2020"] - idx["built_2015"])).round(2)
idx["built_growth_large"] = (idx["year"] <= OLD_SCENE_YEAR) & (idx["built_growth_pp"] > GROWTH_PP)
old = idx[idx["year"] <= OLD_SCENE_YEAR]
print(f"(2) GHSL built-up 2015 -> 2020 for {len(cells)} cells ({len(todo)} new from Earth Engine)")
print(f"    scenes up to {OLD_SCENE_YEAR} ({len(old)} pairs): growth median {old['built_growth_pp'].median():.2f} pp, 90th percentile "
      f"{old['built_growth_pp'].quantile(0.9):.2f} pp, max {old['built_growth_pp'].max():.2f} pp | built_growth_large (> {GROWTH_PP} pp): "
      f"{int(idx['built_growth_large'].sum())} pairs {idx.loc[idx['built_growth_large'], 'dataset_id'].tolist()}")

# ---- 3) split by cell (stable hash; cells keep their group when pairs are added) ----
idx["block_1deg"] = np.floor(idx["cell_lon"]).astype(int).astype(str) + "_" + np.floor(idx["cell_lat"]).astype(int).astype(str)
u = idx[SPLIT_BY].astype(str).map(lambda k: int(hashlib.md5(k.encode()).hexdigest()[:8], 16) / 16 ** 8)
idx["split"] = np.select([u < SPLIT_FRAC[0], u < SPLIT_FRAC[0] + SPLIT_FRAC[1]], ["train", "val"], "test")
assert (idx.groupby("cell_id")["split"].nunique() == 1).all(), "a cell is in two splits"
cu = idx.drop_duplicates("cell_id")[["cell_id", "cell_lon", "cell_lat", "split"]].reset_index(drop=True)
dlon = np.abs(cu["cell_lon"].values[:, None] - cu["cell_lon"].values[None, :]); dlat = np.abs(cu["cell_lat"].values[:, None] - cu["cell_lat"].values[None, :])
near = (np.maximum(dlon, dlat) < 0.1) & (cu["split"].values[:, None] != cu["split"].values[None, :])
print(f"(3) split by {SPLIT_BY} ({SPLIT_FRAC}): {idx[SPLIT_BY].nunique()} groups | neighbouring cells (< 0.1 deg) in different splits: "
      f"{int(near.sum() // 2)} pairs of cells")
print("    pairs per split and sensor:\n" + pd.crosstab(idx["sensor"], idx["split"], margins=True).to_string())
print("    cells per split:", idx.drop_duplicates("cell_id")["split"].value_counts().to_dict())
print("    pairs per split and climate group:\n" + pd.crosstab(idx["climate_group"].fillna("unknown"), idx["split"]).to_string())
shared = idx.groupby("cell_id")["sensor"].nunique(); print(f"    cells seen by more than one sensor: {int((shared > 1).sum())}")

# ---- 4) save ----
idx = idx.sort_values(["sensor", "pair_no"]).reset_index(drop=True)
idx.to_csv(f"{FINAL}/dataset_index.csv", index=False)
print(f"\nSAVED {FINAL}/dataset_index.csv: {len(idx)} rows x {idx.shape[1]} columns")
print("flags among all pairs: blue", int(idx["flag_blue"].sum()), "| red", int(idx["flag_red"].sum()), "| swir1", int(idx["flag_swir1"].sum()),
      "| haze_mild", int(idx["haze_mild"].sum()), "| large_shift", int(idx["large_shift"].sum()))
if problems: print("\nPROBLEMS:", *problems[:30], sep="\n  ")

## B2 EDA (exploratory data analysis)
Reads the index and a random sample of 1,500 clear pixels per pair (cached in `eda/pixel_sample.parquet`; rebuilt when pairs change).
Six figures, saved as PNG in `FINAL-DATASET/eda/`, and two summary tables (`eda_by_sensor.csv`, `eda_by_climate.csv`):

1. `fig1_map.png`: where the pairs are (one dot per cell, size = number of pairs)
2. `fig2_composition.png`: pairs by climate group, year and split, per sensor
3. `fig3_lst.png`: mean LST per pair by climate group and by sensor
4. `fig4_lst_vs_indices.png`: pixel LST anomaly (LST minus the pair's median) against NDVI and NDBI
5. `fig5_rho_per_pair.png`: LST–NDVI and LST–NDBI correlation of every pair
6. `fig6_lst_by_landcover.png`: LST anomaly by WorldCover class (built-up vs tree vs water)

**What to expect:** LST anomaly falls with NDVI and rises with NDBI; built-up pixels warmer than the pair median, trees and water
cooler. These are the urban-heat relationships the models of Members 2 and 3 should learn.

In [ ]:
# B2. EDA: figures and tables across sensors, climate groups and splits -> FINAL-DATASET/eda/ (only reads the pairs)
import os, glob, warnings
import numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import spearmanr
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; FINAL = f"{BASE}/FINAL-DATASET"; EDA = f"{FINAL}/eda"; os.makedirs(EDA, exist_ok=True)
pd.set_option("display.width", 220)
N_PX, REBUILD = 1500, False        # pixels sampled per pair | True = rebuild the pixel sample
SENS = ["HYPERION", "EMIT", "ENMAP"]
SENS_COL = {"HYPERION": "#2a78d6", "EMIT": "#eb6834", "ENMAP": "#1baf7a"}     # fixed colour per sensor in every figure
CLIM = ["tropical", "arid", "temperate", "cold", "polar", "unknown"]
WC_NAMES = {10: "tree", 20: "shrub", 30: "grass", 40: "crop", 50: "built", 60: "bare", 70: "snow", 80: "water",
            90: "wetland", 95: "mangrove", 100: "moss"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "font.size": 10, "legend.frameon": False})
idx = pd.read_csv(f"{FINAL}/dataset_index.csv", dtype={"cell_id": str})
idx["climate_group"] = idx["climate_group"].fillna("unknown")
have = idx[idx["aux_file"].notna()]
print(f"index: {len(idx)} pairs | with aux file: {len(have)}")

# ---- pixel sample (cached) ----
sp = f"{EDA}/pixel_sample.parquet"
pix = pd.read_parquet(sp) if os.path.exists(sp) and not REBUILD else None
if pix is None or set(pix["dataset_id"].unique()) != set(have["dataset_id"]):
    def sample(r):
        d = f"{BASE}/{r['rel_dir']}"
        with rasterio.open(f"{d}/aux_layers.tif") as a:
            names = list(a.descriptions); A = {n: a.read(names.index(n) + 1) for n in ("NDVI", "NDBI", "NDWI", "landsat_clear", "built_share", "landcover_worldcover")}
        with rasterio.open(f"{d}/{r['landsat_file']}") as l: lst = l.read(7)
        ok = (A["landsat_clear"] == 1) & np.isfinite(A["NDVI"]) & np.isfinite(lst) & (lst > 200) & (lst < 360)
        ii = np.flatnonzero(ok)
        if len(ii) < 50: return None
        ii = np.random.default_rng(int(r["pair_no"])).choice(ii, min(N_PX, len(ii)), replace=False); med = float(np.median(lst[ok]))
        g = lambda x: x.ravel()[ii]
        return pd.DataFrame({"dataset_id": r["dataset_id"], "sensor": r["sensor"], "climate_group": r["climate_group"], "split": r["split"],
                             "lst_c": g(lst) - 273.15, "lst_anom": g(lst) - med, "ndvi": g(A["NDVI"]), "ndbi": g(A["NDBI"]),
                             "ndwi": g(A["NDWI"]), "built_share": g(A["built_share"]), "wc": g(A["landcover_worldcover"])})
    with ThreadPoolExecutor(8) as ex: parts = list(ex.map(sample, [r for _, r in have.iterrows()]))
    pix = pd.concat([p for p in parts if p is not None], ignore_index=True); pix.to_parquet(sp, index=False)
    print(f"pixel sample rebuilt: {len(pix):,} pixels from {pix['dataset_id'].nunique()} pairs")
else: print(f"pixel sample from cache: {len(pix):,} pixels")
save = lambda fig, name: (fig.savefig(f"{EDA}/{name}", dpi=150, bbox_inches="tight"), plt.show())
def grouped(ax, tab, title):       # one bar per sensor inside each category
    cats, w = list(tab.index), 0.8 / len(SENS)
    for k, s in enumerate(SENS):
        if s in tab.columns:
            ax.bar(np.arange(len(cats)) + (k - (len(SENS) - 1) / 2) * w, tab[s].values, width=w * 0.88, color=SENS_COL[s], label=s)
    ax.set_xticks(range(len(cats))); ax.set_xticklabels([str(c) for c in cats]); ax.set_title(title, loc="left"); ax.grid(axis="x", visible=False)

# ---- fig 1: map ----
fig, ax = plt.subplots(figsize=(13, 5.6))
ax.axhspan(-23.44, 23.44, color="#f0efec", zorder=0); ax.axhline(0, color="#c3c2b7", lw=0.8)
for s in SENS:
    g = idx[idx["sensor"] == s].groupby(["cell_lon", "cell_lat"]).size().reset_index(name="n")
    ax.scatter(g["cell_lon"], g["cell_lat"], s=14 + 12 * g["n"], color=SENS_COL[s], alpha=0.8, edgecolor="#fcfcfb", linewidth=0.8,
               label=f"{s}: {int((idx['sensor'] == s).sum())} pairs in {len(g)} cells", zorder=3)
ax.set_xlim(-180, 180); ax.set_ylim(-60, 75); ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"Where the {len(idx)} pairs are (dot = cell, size = pairs in the cell; shaded = tropics)", loc="left"); ax.legend(loc="lower left")
save(fig, "fig1_map.png")

# ---- fig 2: composition ----
fig, axs = plt.subplots(1, 3, figsize=(17, 4.4), gridspec_kw={"width_ratios": [4, 6, 3]})
grouped(axs[0], pd.crosstab(idx["climate_group"], idx["sensor"]).reindex([c for c in CLIM if c in set(idx["climate_group"])]), "pairs by climate group (Koppen)")
grouped(axs[1], pd.crosstab(idx["year"], idx["sensor"]), "pairs by scene year")
grouped(axs[2], pd.crosstab(idx["split"], idx["sensor"]).reindex(["train", "val", "test"]), "pairs by split (by cell)")
axs[0].legend(); axs[0].set_ylabel("pairs"); plt.tight_layout(); save(fig, "fig2_composition.png")

# ---- fig 3: LST per pair ----
fig, axs = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [5, 3]}, sharey=True)
rng = np.random.default_rng(1)
for ax, col, order in ((axs[0], "climate_group", [c for c in CLIM if c in set(idx["climate_group"])]), (axs[1], "sensor", SENS)):
    data = [idx.loc[idx[col] == c, "lst_c_mean"].dropna() for c in order]
    ax.boxplot(data, widths=0.5, patch_artist=True, showfliers=False, medianprops={"color": INK, "lw": 1.5},
               boxprops={"facecolor": GRID, "edgecolor": "#8a8984"}, whiskerprops={"color": "#8a8984"}, capprops={"color": "#8a8984"})
    for k, c in enumerate(order):
        g = idx[idx[col] == c]
        for s in SENS:
            v = g.loc[g["sensor"] == s, "lst_c_mean"].dropna()
            ax.scatter(k + 1 + rng.uniform(-0.17, 0.17, len(v)), v, s=11, color=SENS_COL[s], alpha=0.75, zorder=3)
    ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels([f"{c}\n(n={len(x)})" for c, x in zip(order, data)])
    ax.grid(axis="x", visible=False)
axs[0].set_ylabel("mean LST of the pair (deg C, clear pixels)"); axs[0].set_title("LST by climate group (dots coloured by sensor)", loc="left")
axs[1].set_title("LST by sensor", loc="left")
h_ = [plt.Line2D([], [], marker="o", ls="", color=SENS_COL[s], label=s) for s in SENS]; axs[0].legend(handles=h_, loc="upper right")
plt.tight_layout(); save(fig, "fig3_lst.png")

# ---- fig 4: pixel LST anomaly vs NDVI / NDBI ----
fig, axs = plt.subplots(1, 2, figsize=(14, 5.2))
rho_px = {}
for ax, col, lab, rngx in ((axs[0], "ndvi", "NDVI (Landsat)", (-0.3, 1.0)), (axs[1], "ndbi", "NDBI (Landsat)", (-0.7, 0.5))):
    p = pix[pix[col].between(*rngx) & pix["lst_anom"].between(-15, 15)]
    hb = ax.hexbin(p[col], p["lst_anom"], gridsize=70, bins="log", cmap="Blues", mincnt=1, extent=(*rngx, -15, 15))
    edges = np.linspace(*rngx, 27); mid = (edges[:-1] + edges[1:]) / 2
    med = p.groupby(pd.cut(p[col], edges), observed=False)["lst_anom"].median().values
    ax.plot(mid, med, color=INK, lw=2, label="median per bin"); ax.axhline(0, color="#8a8984", lw=0.8)
    rho_px[col] = spearmanr(p[col], p["lst_anom"])[0]
    ax.set_title(f"LST anomaly vs {lab}: Spearman rho = {rho_px[col]:.2f} ({len(p):,} pixels)", loc="left")
    ax.set_xlabel(lab); ax.set_ylabel("LST minus the pair's median (K)"); ax.legend(loc="upper right")
    fig.colorbar(hb, ax=ax, label="pixels (log scale)")
plt.tight_layout(); save(fig, "fig4_lst_vs_indices.png")

# ---- fig 5: correlation per pair ----
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
for ax, col, lab in ((axs[0], "rho_lst_ndvi", "LST vs NDVI"), (axs[1], "rho_lst_ndbi", "LST vs NDBI")):
    for s in SENS:
        v = idx.loc[idx["sensor"] == s, col].dropna()
        if len(v): ax.hist(v, bins=np.linspace(-1, 1, 41), histtype="step", lw=2, color=SENS_COL[s], label=f"{s} (median {v.median():.2f})")
    ax.axvline(0, color="#8a8984", lw=0.8); ax.set_xlabel("Spearman rho within the pair (clear pixels)")
    ax.set_title(f"{lab}: one value per pair", loc="left"); ax.legend(loc="upper left")
axs[0].set_ylabel("pairs"); plt.tight_layout(); save(fig, "fig5_rho_per_pair.png")

# ---- fig 6: LST anomaly by land-cover class ----
pw = pix[pix["wc"].notna()].copy(); pw["class"] = pw["wc"].round().astype(int).map(WC_NAMES)
t6 = pw.groupby("class")["lst_anom"].agg(median="median", q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75), n="size")
t6 = t6[t6["n"] >= 500].sort_values("median")
fig, ax = plt.subplots(figsize=(10, 0.45 * len(t6) + 1.4))
y = np.arange(len(t6))
ax.barh(y, t6["median"], color="#8a8984", height=0.6)
ax.errorbar(t6["median"], y, xerr=[t6["median"] - t6["q25"], t6["q75"] - t6["median"]], fmt="none", ecolor=INK, elinewidth=1, capsize=3)
ax.set_yticks(y); ax.set_yticklabels([f"{c} (n={n:,})" for c, n in zip(t6.index, t6["n"])]); ax.axvline(0, color=INK2, lw=0.8)
ax.set_xlabel("LST minus the pair's median (K): bar = median, line = 25-75 %"); ax.grid(axis="y", visible=False)
ax.set_title("LST anomaly by ESA WorldCover class (all pairs)", loc="left"); plt.tight_layout(); save(fig, "fig6_lst_by_landcover.png")

# ---- tables ----
def table(by):
    t = idx.groupby(by).agg(pairs=("dataset_id", "size"), cells=("cell_id", "nunique"), year_min=("year", "min"), year_max=("year", "max"),
                            lst_c=("lst_c_mean", "median"), ndvi=("ndvi_mean", "median"), ndbi=("ndbi_mean", "median"),
                            built_share=("built_share_mean", "median"), wc_built=("wc_built", "median"), elev_m=("elev_mean", "median"),
                            rho_lst_ndvi=("rho_lst_ndvi", "median"), rho_lst_ndbi=("rho_lst_ndbi", "median"),
                            haze_mild=("haze_mild", "sum"), flag_swir1=("flag_swir1", "sum"))
    return t.round(3)
tb_s, tb_c = table("sensor"), table("climate_group")
tb_s.to_csv(f"{EDA}/eda_by_sensor.csv"); tb_c.to_csv(f"{EDA}/eda_by_climate.csv"); t6.round(3).to_csv(f"{EDA}/eda_lst_by_landcover.csv")
print("\nby sensor:\n" + tb_s.to_string()); print("\nby climate group:\n" + tb_c.to_string())
print("\nKEY NUMBERS")
print(f"  pixel level: Spearman LST anomaly vs NDVI {rho_px['ndvi']:.2f}, vs NDBI {rho_px['ndbi']:.2f}")
print(f"  pair level: LST-NDVI rho < 0 in {100 * (idx['rho_lst_ndvi'] < 0).mean():.0f} % of pairs, LST-NDBI rho > 0 in {100 * (idx['rho_lst_ndbi'] > 0).mean():.0f} %")
if "built" in t6.index: print(f"  built-up pixels: {t6.loc['built', 'median']:+.1f} K vs the pair median" +
                              (f" | tree {t6.loc['tree', 'median']:+.1f} K" if "tree" in t6.index else "") +
                              (f" | water {t6.loc['water', 'median']:+.1f} K" if "water" in t6.index else ""))
print(f"\nfigures: {sorted(os.path.basename(f) for f in glob.glob(f'{EDA}/*.png'))}")

## B3 Handover for Member 2
Writes `FINAL-DATASET/README_DATASET.txt` (what is where, how to load a pair, the split, caveats) and `dataset_index_columns.csv`
(meaning of every column), then prints a short message you can send to Member 2. Safe to re-run (overwrites only these two files).

In [ ]:
# B3. HANDOVER: README_DATASET.txt + dataset_index_columns.csv in FINAL-DATASET/ + message for Member 2. Safe to re-run.
import os, glob, time
import pandas as pd

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; FINAL = f"{BASE}/FINAL-DATASET"
idx = pd.read_csv(f"{FINAL}/dataset_index.csv", dtype={"cell_id": str})

# ---- 1) column dictionary ----
COLS = {"dataset_id": "unique pair ID: SENSOR_pairno", "sensor": "HYPERION, EMIT or ENMAP", "pair_no": "pair number in the sensor's pairs/ archive",
        "rel_dir": "pair folder relative to MEMBER1___PREPROCESSING/", "reflectance_file": "hyperspectral surface reflectance x 10000 (int16, nodata -32768)",
        "landsat_file": "Landsat 8/9 on the same grid: blue, green, red, nir, swir1, swir2 (reflectance), LST_K (kelvin), QA_PIXEL or clear_frac",
        "aux_file": "aux_layers.tif: 12 layers on the same grid (see README)", "res_m": "nominal pixel size (m)", "crs": "coordinate system of the pair grid",
        "width": "pixels", "height": "pixels", "n_bands": "hyperspectral bands kept", "scene_id": "hyperspectral scene ID",
        "scene_date": "hyperspectral acquisition date", "year": "scene year", "month": "scene month", "landsat_scene": "Landsat scene ID",
        "landsat_dt_days": "Landsat date minus hyperspectral date (days)", "cell_id": "screening cell (0.07 deg); split groups are built on it",
        "cell_lon": "cell centre longitude", "cell_lat": "cell centre latitude", "profile": "screening profile",
        "koppen_code": "Koppen-Geiger class 1991-2020 (Beck et al.)", "climate_group": "tropical / arid / temperate / cold / polar",
        "qc_version": "quality-control version (v2.3 for all)", "coverage_joint": "share of the cell valid in both sensors",
        "landsat_clear_frac_qc": "Landsat clear share inside the hyperspectral footprint (QC)", "ndvi_r_after": "NDVI correlation hyperspectral vs Landsat after alignment",
        "ndvi_gain": "mean hyperspectral NDVI / mean Landsat NDVI over vegetation (haze test)", "aod550": "aerosol optical depth at 550 nm",
        "sun_elev": "sun elevation (deg)", "across_offnadir": "EnMAP across-track viewing angle (deg; empty for other sensors)",
        "flag_blue": "hyperspectral/Landsat blue ratio outside 0.85-1.15 (flag only)", "flag_red": "same for red", "flag_swir1": "same for SWIR1",
        "haze_mild": "NDVI gain 0.80-0.85 (slight haze; passed)", "haze_unchecked": "too little vegetation for the haze test",
        "large_shift": "alignment shift > 3 px (confirmed)", "ghsl_epoch": "GHSL epoch used for built_share",
        "n_px": "pixels in the grid", "n_joint_px": "pixels clear in Landsat and valid in the hyperspectral file", "joint_frac": "n_joint_px / n_px",
        "landsat_clear_frac": "Landsat clear share of the grid (aux)", "elev_mean": "mean SRTM elevation (m)", "slope_mean": "mean slope (deg)",
        "built_share_mean": "mean GHSL built-up share (0-1)", "dw_valid_frac": "share of pixels with a Dynamic World class",
        "lst_c_mean": "mean LST (deg C) over joint pixels", "lst_c_p10": "10th percentile LST (deg C)", "lst_c_p90": "90th percentile LST (deg C)",
        "ndvi_mean": "mean Landsat NDVI (joint pixels)", "ndbi_mean": "mean Landsat NDBI", "ndwi_mean": "mean Landsat NDWI",
        "ndvi_hs_mean": "mean hyperspectral NDVI", "rho_lst_ndvi": "Spearman rho LST vs NDVI inside the pair", "rho_lst_ndbi": "Spearman rho LST vs NDBI inside the pair",
        "built_2015": "GHSL built-up share of the cell, 2015", "built_2020": "GHSL built-up share of the cell, 2020",
        "built_growth_pp": "built-up change 2015 -> 2020 (percentage points)", "built_growth_large": "scene <= 2017 and growth > 10 pp (cell changed after the scene)",
        "block_1deg": "1 x 1 deg block (for a stricter spatial split)", "split": "train / val / test, assigned by cell"}
for c in idx.columns:
    if c.startswith("wc_"): COLS[c] = f"share of the cell in WorldCover class '{c[3:]}'"
pd.DataFrame([{"column": c, "meaning": COLS.get(c, "")} for c in idx.columns]).to_csv(f"{FINAL}/dataset_index_columns.csv", index=False)

# ---- 2) README ----
by = idx.groupby("sensor").agg(pairs=("dataset_id", "size"), cells=("cell_id", "nunique"), res=("res_m", "first"), bands=("n_bands", "median"),
                               y0=("year", "min"), y1=("year", "max"))
sp = pd.crosstab(idx["sensor"], idx["split"]).reindex(columns=["train", "val", "test"], fill_value=0)
lines = "\n".join(f"  {s:<9} {int(r['pairs']):>4} pairs  {int(r['cells']):>4} cells  {int(r['res'])} m  {int(r['bands'])} bands  {int(r['y0'])}-{int(r['y1'])}"
                  f"   split train/val/test {sp.loc[s, 'train']}/{sp.loc[s, 'val']}/{sp.loc[s, 'test']}" for s, r in by.iterrows())
yrs = "; ".join(f"{s} {int(r['y0'])}-{int(r['y1'])}" for s, r in by.iterrows())
dt = idx.assign(adt=pd.to_numeric(idx["landsat_dt_days"], errors="coerce").abs()).groupby("sensor")["adt"].agg(["median", "max"])
dts = "; ".join(f"{s} median {r['median']:.2f} d, max {r['max']:.0f} d" for s, r in dt.iterrows())
readme = f"""FINAL DATASET: hyperspectral + Landsat 8/9 pairs for urban heat analysis (Member 1 -> Member 2)
=================================================================================================
Made {time.strftime('%Y-%m-%d')} by notebook FINAL_DATASET.ipynb. {len(idx)} pairs, {idx['cell_id'].nunique()} cells, QC v2.3 for all.

{lines}

WHERE THINGS ARE (paths relative to MEMBER1___PREPROCESSING/; column rel_dir of dataset_index.csv)
  <SENSOR>-LANDSAT/PASSED-PAIRS/<N>/
      <sensor>_reflectance.tif  hyperspectral surface reflectance x 10000 (int16), nodata -32768; wavelength in each band description
      landsat_<id>.tif          Landsat 8/9 on the SAME grid (float32, NaN = no data): blue, green, red, nir, swir1, swir2 (reflectance),
                                LST_K (land surface temperature, kelvin), band 8 = QA_PIXEL (Hyperion) or clear_frac (EMIT, EnMAP)
      aux_layers.tif            12 layers on the SAME grid (float32, NaN = no data):
                                1 NDVI  2 NDBI  3 NDWI (McFeeters)  4 MNDWI  (Landsat, clear pixels only)
                                5 NDVI_hs (from the hyperspectral bands)  6 landsat_clear (1 = clear, same rule as the QC)
                                7 elevation_m  8 slope_deg  9 aspect_deg (SRTM)  10 built_share (GHSL, 0-1, epoch nearest the scene year, <= 2020)
                                11 landcover_worldcover (ESA WorldCover 2021 class code)  12 landcover_dw (Dynamic World, scene date +/- 6 months)
      pair_info.json            scene IDs, dates, angles, atmosphere, climate, all QC numbers
  <SENSOR>-LANDSAT/PASSED-PAIRS/README.txt, handover_pass.csv, <sensor>_bands.json   per-sensor details and band lists
  FINAL-DATASET/dataset_index.csv          one row per pair: files, dates, cell, climate, QC numbers, layer statistics, split
  FINAL-DATASET/dataset_index_columns.csv  meaning of every column
  FINAL-DATASET/eda/                       figures and tables

HOW TO LOAD ONE PAIR (Python)
  import pandas as pd, numpy as np, rasterio
  BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"
  idx = pd.read_csv(f"{{BASE}}/FINAL-DATASET/dataset_index.csv"); r = idx.iloc[0]; d = f"{{BASE}}/{{r['rel_dir']}}"
  with rasterio.open(f"{{d}}/{{r['reflectance_file']}}") as s: hs = s.read().astype("float32"); hs[hs == -32768] = np.nan; hs /= 10000
  with rasterio.open(f"{{d}}/{{r['landsat_file']}}") as s: ls = s.read()          # ls[6] = LST in kelvin
  with rasterio.open(f"{{d}}/{{r['aux_file']}}") as s: aux = s.read(); names = s.descriptions
  valid = np.isfinite(hs).all(axis=0) & (aux[names.index("landsat_clear")] == 1)   # pixels usable for training

SPLIT (column split)
  Assigned by CELL (a stable hash of cell_id, about 70/15/15 % of cells): all dates of a cell are in the same split, so the test set has
  only places the model never saw. Use this column; do not split by pair or by pixel. For a stricter spatial test use block_1deg.

THINGS TO KNOW
  - Resolution differs: Hyperion and EnMAP 30 m, EMIT 60 m (lat/lon grid). Band sets differ per sensor (see <sensor>_bands.json);
    resample spectrally to a common set or train per sensor.
  - Scene years: {yrs}. Hyperion cells are mostly NOT Kerala-like (dense city / semi-arid);
    EMIT and EnMAP give the humid tropical cells. Filter on climate_group or profile if needed.
  - Flags (still PASS): flag_blue / flag_red / flag_swir1 = band ratio to Landsat outside 0.85-1.15; haze_mild = slight haze.
    Use them to drop or down-weight pairs for the cleanest set: {int(idx['haze_mild'].sum())} haze_mild, {int(idx['flag_swir1'].sum())} flag_swir1.
    flag_blue is very common ({int(idx['flag_blue'].sum())} of {len(idx)}): the blue band differs between the sensors' atmospheric
    corrections; it is not a reason to drop a pair.
  - SRTM is a surface model: over forests and dense buildings elevation_m includes part of the canopy / building height.
  - Split shares are by cell and random, so they differ a little from 70/15/15 ({int((idx['split'] == 'train').sum())}/{int((idx['split'] == 'val').sum())}/{int((idx['split'] == 'test').sum())} pairs);
    for cross-validation group by cell_id (e.g. sklearn GroupKFold).
  - Known sensor differences: Hyperion SWIR reads lower than Landsat (about 0.88); EnMAP SWIR higher (about 1.16-1.19);
    EnMAP NIR depends on the viewing angle (BRDF; across_offnadir).
  - LST is Landsat Collection 2 Level-2 surface temperature. Time between the Landsat and the hyperspectral scene
    (|landsat_dt_days|): {dts}.
  - All EMIT and EnMAP pairs are tropical; the arid, temperate and cold pairs are all Hyperion (2013-2016).
  - Built-up change: {int(idx['built_growth_large'].sum())} pairs flagged built_growth_large (cell changed between the scene and 2020).
  - Dynamic World starts in June 2015: landcover_dw is empty for most Hyperion pairs; use landcover_worldcover (2021) there.
  - PASS means "passed these checks", not perfect data.
"""
open(f"{FINAL}/README_DATASET.txt", "w").write(readme)
size = sum(os.path.getsize(os.path.join(dp, f)) for s_ in ("HYPERION", "EMIT", "ENMAP")
           for dp, _, fs in os.walk(f"{BASE}/{s_}-LANDSAT/PASSED-PAIRS") if "_to_delete" not in dp for f in fs) / 1e9
print(f"WRITTEN: {FINAL}/README_DATASET.txt and dataset_index_columns.csv ({idx.shape[1]} columns described)")
print(f"dataset size (three PASSED-PAIRS folders incl. aux layers): {size:.2f} GB | figures: {len(glob.glob(f'{FINAL}/eda/*.png'))}")
print("\n" + "=" * 100 + "\nMESSAGE FOR MEMBER 2 (copy):\n")
print(f"""Hi, the Member 1 dataset is ready on the shared Drive: MEMBER1___PREPROCESSING/
- {len(idx)} hyperspectral + Landsat 8/9 pairs ({', '.join(f'{s} {n}' for s, n in idx['sensor'].value_counts().items())}), {idx['cell_id'].nunique()} cells, all QC v2.3.
- Start with FINAL-DATASET/dataset_index.csv (one row per pair, file paths in rel_dir) and FINAL-DATASET/README_DATASET.txt.
- Each pair: hyperspectral reflectance, Landsat bands + LST (target), and aux_layers.tif (NDVI, NDBI, NDWI, MNDWI, elevation, slope,
  built-up share, land cover), all on the same grid.
- Please use the split column (train/val/test by cell) so test places are never seen in training.
- Caveats are in the README (resolution 30 vs 60 m, different band sets per sensor, flags).""")

# Design choices, evidence and references

| Choice | Reason / evidence | Source |
|---|---|---|
| Indices from each pair's **own** Landsat bands | Same date and grid as the hyperspectral scene and the LST, so no time mismatch | Design decision |
| NDVI, NDBI, NDWI (McFeeters), MNDWI (Xu) | Standard vegetation, built-up and water indices in urban-heat studies; MNDWI separates water from built-up better than NDWI | Rouse et al. 1974; Zha et al. 2003; McFeeters 1996; Xu 2006 (check before citing) |
| Indices only on clear Landsat pixels; clear mask delivered as a band | Cloud and shadow would give false indices; the same clear rule as QC v2.3 | Design decision |
| NDVI also from the hyperspectral bands | Checks the two sensors against each other and gives an NDVI where Landsat is not clear | Design decision |
| SRTM 1 arc-second, slope / aspect computed on each pair grid | Static terrain; elevation and slope affect LST (lapse rate, illumination) | NASA SRTM (Farr et al. 2007) |
| GHSL built-up share, epoch nearest the scene year, at most 2020 | Built-up density is the main urban-heat driver; GHSL epochs after 2020 are projections | JRC GHSL P2023A (Pesaresi et al. 2023; check) |
| ESA WorldCover 2021 for every pair | One consistent 10 m class scheme (the one used in the screening); most frequent class per pixel | Zanaga et al. 2022 |
| Dynamic World label mode within ± 6 months of the scene | Year-matched land cover where Sentinel-2 exists (from June 2015) | Brown et al. 2022 |
| GHSL change 2015 → 2020 per cell, flag > 10 pp for scenes up to 2017 | Quantifies the screening's time mismatch instead of redoing the screening | Design decision (threshold) |
| Split by cell, stable hash, 70 / 15 / 15 | Dates of one cell are strongly correlated; a hash keeps groups fixed when pairs are added | Design decision |
| Aux file next to the pair files, same grid | Member 2 loads one folder per pair; no resampling needed | Design decision |

## Results (run of 24 Oct 2026)
- 422 pairs in 281 cells; aux layers for all 422 (A1, 6.4 min, no failures). Hyperspectral vs Landsat NDVI across pairs r = 0.976.
- Inside the pairs LST falls with NDVI (median Spearman −0.56; negative in 96 % of pairs) and rises with NDBI (median +0.62 to +0.68;
  positive in 99 %), equally for all three sensors. Note: LST, NDVI and NDBI all come from Landsat, so this shows the relationship
  holds across places, years and climates; it is not a comparison of the hyperspectral sensors.
- Pooled pixels (LST minus the pair median): Spearman −0.35 with NDVI, +0.44 with NDBI (weaker than inside pairs because the
  strength of the contrast differs from pair to pair).
- By WorldCover class (median LST anomaly): built-up +0.7 K, tree −0.8 K, water −5.4 K. The built-up value is small because the pair
  median is itself mostly built-up (urban cells); built-up vs tree differ by about 1.5 K.
- Median LST: arid 37.1 °C, tropical 35.5, cold 35.9 (summer scenes), temperate 29.2. All EMIT and EnMAP pairs are tropical.
- GHSL built-up growth 2015 → 2020 in the Hyperion cells: median 0.45, max 2.08 percentage points → no pair flagged.
- Split: 321 / 45 / 56 pairs (213 / 33 / 35 cells).

## Limits
- WorldCover (2021) is years after the Hyperion scenes (2013–2016); Dynamic World covers only part of them; the GHSL check measures the effect.
- GHSL is 100 m, resampled bilinearly to 30 / 60 m: a smooth density, not building outlines.
- SRTM is a surface model: over forests and dense buildings `elevation_m` includes part of the canopy / building height (visible in the A1 figure, EnMAP 26).
- GHSL check (B1, 24 Oct): built-up growth 2015 → 2020 in the Hyperion cells is small (median 0.45, max 2.08 percentage points), so the
  screening's time mismatch hardly matters for built-up density; no pair flagged.
- The split by cell gave 321 / 45 / 56 pairs (213 / 33 / 35 cells), a little off 70 / 15 / 15 by chance; 14 pairs of neighbouring cells
  lie in different splits (use `block_1deg` for a stricter test); the 18 cold-climate pairs have no validation pair.
- EMIT pairs are on a 60 m latitude/longitude grid; terrain and land cover are sampled on that grid.
- LST is Landsat's Collection 2 product (single-channel algorithm, ASTER GED emissivity); no own LST retrieval.
- The split is random by cell, not stratified; the per-sensor and per-climate shares are printed in B1.

## References (check before citing)
- Brown, C. F., et al. (2022). Dynamic World, near real-time global 10 m land use land cover mapping. *Scientific Data* 9, 251.
- Farr, T. G., et al. (2007). The Shuttle Radar Topography Mission. *Reviews of Geophysics* 45, RG2004.
- McFeeters, S. K. (1996). The use of the Normalized Difference Water Index (NDWI). *International Journal of Remote Sensing* 17(7).
- Pesaresi, M., et al. (2023). GHS-BUILT-S R2023A. European Commission, Joint Research Centre.
- Rouse, J. W., et al. (1974). Monitoring vegetation systems in the Great Plains with ERTS. NASA SP-351.
- Xu, H. (2006). Modification of normalised difference water index (MNDWI). *International Journal of Remote Sensing* 27(14).
- Zanaga, D., et al. (2022). ESA WorldCover 10 m 2021 v200. Zenodo.
- Zha, Y., Gao, J., Ni, S. (2003). Use of normalized difference built-up index in automatically mapping urban areas from TM imagery. *IJRS* 24(3).
- USGS Landsat Collection 2 Level-2 Science Product Guide (surface reflectance and surface temperature scale factors, QA_PIXEL).